In [0]:
# Reading Customer Data
customers_df = spark.read.csv("/Volumes/databricks_incremental_medallion_lakehouse/default/incremental_load_files/customers.csv", header=True, inferSchema=True)

In [0]:
customers_df.display()

In [0]:
# Reading orders data
orders_df = spark.read.csv("/Volumes/databricks_incremental_medallion_lakehouse/default/incremental_load_files/orders.csv", header=True, inferSchema=True)

In [0]:
orders_df.display()


In [0]:
# Reading Products Data

products_df = spark.read.csv("/Volumes/databricks_incremental_medallion_lakehouse/default/incremental_load_files/products.csv", header=True, inferSchema=True)


In [0]:
products_df.display()

#Add injection date

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
customers_df = customers_df.withColumn(
    "ingestion_date",
    F.current_timestamp()
)

In [0]:
orders_df = orders_df.withColumn(
    "ingestion_date",
    F.current_timestamp())

In [0]:
products_df = products_df.withColumn(
    "ingestion_date",
    F.current_timestamp())

#Convert updated_at to **timestamp**

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
customers_df = customers_df.withColumn(
    "updated_at",
    F.to_timestamp("updated_at","dd/MM/yyyy HH:mm"))

In [0]:
orders_df = orders_df.withColumn(
    "updated_at",
    F.to_timestamp("updated_at","dd/MM/yyyy HH:mm"))

In [0]:
products_df = products_df.withColumn(
    "updated_at",
    F.to_timestamp("updated_at","dd/MM/yyyy HH:mm"))

#one time we need to overwritr the data

In [0]:
#customers_df.write \
 #   .format("delta") \
  #  .mode("overwrite") \
   # .saveAsTable("databricks_incremental_medallion_lakehouse.bronze.customers_bronze")

In [0]:
#orders_df.write \
 #   .format("delta") \
  #  .mode("overwrite") \
   # .saveAsTable("databricks_incremental_medallion_lakehouse.bronze.orders_bronze")

In [0]:
#products_df.write \
 #   .format("delta") \
  #  .mode("overwrite") \
   # .saveAsTable("databricks_incremental_medallion_lakehouse.bronze.products_bronze")

#Incremental Merge Customers

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

target_table = "databricks_incremental_medallion_lakehouse.bronze.customers_bronze"

# Get Delta target table
target = DeltaTable.forName(spark, target_table)

# Incremental MERGE
(
    target.alias("target")
    .merge(
        customers_df.alias("source"),
        "target.customer_id = source.customer_id"
    )

    # Existing customer -> UPDATE
    .whenMatchedUpdate(
        condition="""
            source.updated_at > target.updated_at
        """,
        set={
            "customer_name": "source.customer_name",
            "city": "source.city",
            "state": "source.state",
            "updated_at": "source.updated_at",
            "ingestion_date": "source.ingestion_date",
            "last_updated_date": "current_timestamp()"
        }
    )

    # New customer -> INSERT
    .whenNotMatchedInsert(
        values={
            "customer_id": "source.customer_id",
            "customer_name": "source.customer_name",
            "city": "source.city",
            "state": "source.state",
            "updated_at": "source.updated_at",
            "ingestion_date": "source.ingestion_date",
            "initial_load_date": "current_timestamp()",
            "last_updated_date": "current_timestamp()"
        }
    )

    .execute()
)

#Incremental Merge Orders

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

target = DeltaTable.forName(
    spark,
    "databricks_incremental_medallion_lakehouse.bronze.orders_bronze"
)

(
    target.alias("target")
    .merge(
        orders_df.alias("source"),
        "target.order_id = source.order_id"
    )
    .whenMatchedUpdate(
        condition="source.updated_at > target.updated_at",
        set={
            "customer_id": "source.customer_id",
            "order_date": "source.order_date",
            "amount": "source.amount",
            "status": "source.status",
            "updated_at": "source.updated_at",
            "ingestion_date": "source.ingestion_date",
            "last_updated_date": "current_timestamp()"
        }
    )
    .whenNotMatchedInsert(
        values={
            "order_id": "source.order_id",
            "customer_id": "source.customer_id",
            "order_date": "source.order_date",
            "amount": "source.amount",
            "status": "source.status",
            "updated_at": "source.updated_at",
            "ingestion_date": "source.ingestion_date",
            "initial_load_date": "current_timestamp()",
            "last_updated_date": "current_timestamp()"
        }
    )
    .execute()
)

#Incremental Mergr products

In [0]:
target = DeltaTable.forName(
    spark,
    "databricks_incremental_medallion_lakehouse.bronze.products_bronze"
)

(
    target.alias("target")
    .merge(
        products_df.alias("source"),
        "target.product_id = source.product_id"
    )
    .whenMatchedUpdate(
        condition="source.updated_at > target.updated_at",
        set={
            "product_name": "source.product_name",
            "category": "source.category",
            "price": "source.price",
            "updated_at": "source.updated_at",
            "ingestion_date": "source.ingestion_date",
            "last_updated_date": "current_timestamp()"
        }
    )
    .whenNotMatchedInsert(
        values={
            "product_id": "source.product_id",
            "product_name": "source.product_name",
            "category": "source.category",
            "price": "source.price",
            "updated_at": "source.updated_at",
            "ingestion_date": "source.ingestion_date",
            "initial_load_date": "current_timestamp()",
            "last_updated_date": "current_timestamp()"
        }
    )
    .execute()
)

In [0]:
%sql
select * from databricks_incremental_medallion_lakehouse.bronze.customers_bronze;

##Create Audit Table for Customers_broze 

In [0]:
%sql
-- CREATE TABLE IF NOT EXISTS databricks_incremental_medallion_lakehouse.     bronze.pipeline_audit
-- (
    -- pipeline_name STRING,
    -- table_name STRING,
    -- start_time TIMESTAMP,
    -- end_time TIMESTAMP,
    -- records_processed BIGINT,
    -- status STRING,
    -- error_message STRING
-- )
-- USING DELTA;

In [0]:
spark.sql("""
INSERT INTO databricks_incremental_medallion_lakehouse.bronze.pipeline_audit
VALUES (
    'Project2_Medallion_Pipeline',
    'customers_bronze',
    current_timestamp(),
    current_timestamp(),
    6,
    'SUCCESS',
    NULL
)
""")

In [0]:
%sql
select * from databricks_incremental_medallion_lakehouse.bronze.pipeline_audit;
  
